# N-Candidate Solver — Concealed Hinges (N=2)

The `NCandidateSolver` solving a **paired-product family** against the real catalog: 53 hinges × 55 mounting plates = 2,915 pairs, 14 constraint rules.

This is the same problem that `engine_v1/solver.py` solves with its dedicated `HingeConstraintEngine`. This notebook proves the generic N-candidate solver produces equivalent results using the same `sample-data/` catalog.

**Compare with:**
- `../v1/v1_hinge_constraint_demo.ipynb` — the V1 dedicated hinge engine (same data, different solver)
- `v2_drawer_slide_demo.ipynb` — N=1 (single product, no Cartesian product)
- `../v2-n-candidate/v2_n_candidate_demo.ipynb` — N=3 (LED lighting, triple Cartesian product)

## Why N=2?

The contractor knows their cabinet (door thickness, height, weight, overlay, boring pattern). They need the engine to find compatible **hinge + plate** pairs. The cabinet is fixed input (requirements), the products are searched.

The solver computes the Cartesian product of hinges × plates (after pre-filtering) and evaluates each pair against 14 rules.

### Rules (14)

| Rule | Category | Constraint |
|---|---|---|
| R001 brand_lock | Hard | Hinge and plate same brand |
| R002 series_compatibility | Hard | Plate lists hinge's series |
| R003 cabinet_type_match | Hard | All agree on frameless/face-frame |
| R004 overlay_in_range | Hard | Desired overlay within plate's range |
| R005 inset_support | Hard | Plate supports inset if needed |
| R006 door_thickness_range | Hard | Door thick enough for hinge |
| R007 door_weight_limit | Hard | Weight ≤ capacity × hinges |
| R009 boring_pattern_match | Hard | Boring pattern matches |
| R013 corner_cabinet_angle | Hard | ≥155° for corner cabinets |
| R012 adjacent_door_clearance | Hard | Combined overlay ≤ partition |
| R011 face_frame_overlay | Hard | Overlay ≤ frame width − 3mm |
| R014 mounting_method_match | Hard | Compatible mounting methods |
| R015 cup_depth | Hard | Door ≥ cup depth + 2mm |
| PREF soft_close | Preference | Soft-close if requested |

**Data:** `sample-data/hinges.json` (53 products), `sample-data/mounting_plates.json` (55 products) — real catalog data.

In [ ]:
import sys, time
from pathlib import Path
from collections import Counter

# Find project root (works regardless of kernel working directory)
_project_root = Path.cwd()
while _project_root != _project_root.parent:
    if (_project_root / "sample-data").exists() and (_project_root / "engine_v2").exists():
        break
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root))
DATA_DIR = _project_root / "sample-data"

from engine_v2.core.solver_n import NCandidateSolver
from engine_v2.families.concealed_hinge.config import HINGE_N_CONFIG
from engine_v2.families.concealed_hinge.loader import load_from_json
from engine_v2.families.concealed_hinge.models import (
    ApplicationType, CabinetPosition, CabinetType, HingeRequirements,
)

hinges, plates = load_from_json(DATA_DIR)

solver = NCandidateSolver(
    config=HINGE_N_CONFIG,
    product_lists={"hinge": hinges, "plate": plates},
)

print(f"Loaded {len(hinges)} hinges and {len(plates)} plates from sample-data/")
print(f"Roles: {HINGE_N_CONFIG.role_names}  (N={len(HINGE_N_CONFIG.roles)})")
print(f"Rules: {len(HINGE_N_CONFIG.rules)}")
print(f"Full Cartesian product: {len(hinges)} x {len(plates)} = {len(hinges) * len(plates):,} pairs")
print(f"Pre-filtering narrows hinges before evaluation (by application, cabinet type, brand)")

## 1. Catalog Overview

In [ ]:
# Hinge distribution
brand_counts = Counter(h.brand for h in hinges)
app_counts = Counter(h.application.value for h in hinges)
angle_counts = Counter(h.opening_angle_deg for h in hinges)

print("HINGE CATALOG SUMMARY")
print(f"  By brand:       {dict(brand_counts)}")
print(f"  By application: {dict(app_counts)}")
print(f"  By angle:       {dict(angle_counts)}")

print(f"\nPLATE CATALOG SUMMARY")
print(f"  By brand:       {dict(Counter(p.brand for p in plates))}")
print(f"  By type:        {dict(Counter(p.plate_type.value for p in plates))}")
print(f"  By cabinet:     {dict(Counter(p.cabinet_type.value for p in plates))}")

In [ ]:
# Full hinge listing
print(f"{'SKU':<22} {'Brand':<8} {'Series':<22} {'App':<14} {'Angle':>5} {'Weight':>7} {'SC':>3} {'Price':>7}")
print("-" * 90)
for h in hinges:
    sc = "Yes" if h.soft_close else "No"
    price = f"${h.price_usd:.2f}" if h.price_usd else "N/A"
    print(f"{h.sku:<22} {h.brand:<8} {h.series.value:<22} {h.application.value:<14} {h.opening_angle_deg:>4}\u00b0 {h.max_door_weight_kg:>5.1f}kg {sc:>3} {price:>7}")

## 2. Customer Scenarios

Five scenarios exercising different parts of the constraint space — the same scenarios used in the V1 demo and the cross-solver consistency tests.

In [ ]:
scenarios = [
    (
        "Scenario 1: Standard kitchen — Blum, full overlay, soft-close",
        HingeRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
            door_weight_kg=5.2, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, preferred_brand="Blum",
        ),
    ),
    (
        "Scenario 2: Corner cabinet — needs wide-angle hinge (>=155\u00b0)",
        HingeRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=800,
            door_weight_kg=4.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, cabinet_position=CabinetPosition.CORNER,
        ),
    ),
    (
        "Scenario 3: Tall pantry door — 1600mm, 14kg, Grass preferred",
        HingeRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=22, door_height_mm=1600,
            door_weight_kg=14.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, preferred_brand="Grass",
        ),
    ),
    (
        "Scenario 4: Adjacent doors — half overlay, shared partition",
        HingeRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
            door_weight_kg=4.0, application=ApplicationType.HALF_OVERLAY, desired_overlay_mm=6,
            boring_pattern_mm=45, soft_close=True, preferred_brand="Blum",
            has_adjacent_door=True, adjacent_door_overlay_mm=6, partition_thickness_mm=19,
        ),
    ),
    (
        "Scenario 5: CONSTRAINT VIOLATION — heavy corner door (800mm, 12kg, Blum)",
        HingeRequirements(
            cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=22, door_height_mm=800,
            door_weight_kg=12.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
            boring_pattern_mm=45, soft_close=True, cabinet_position=CabinetPosition.CORNER,
            preferred_brand="Blum",
        ),
    ),
]

for title, req in scenarios:
    result = solver.solve_with_explanation(req)
    
    print("=" * 80)
    print(f"  {title}")
    print(f"  Status: {result['status']}")
    print(f"  {result['message']}")
    
    if result["status"] == "solved":
        rec = result["recommended"]
        hinge_sku = rec["candidates"]["hinge"]["sku"]
        plate_sku = rec["candidates"]["plate"]["sku"]
        price = f"${rec['total_price_usd']:.2f}" if rec['total_price_usd'] else "N/A"
        print(f"\n  Recommended: {hinge_sku} + {plate_sku}")
        print(f"  Hinges/door: {rec['derived'].get('hinges_per_door', '?')}  |  Price: {price}/door")
        print(f"  + {len(result['alternatives'])} alternative(s)")
    
    elif result["status"] == "no_solution":
        print(f"\n  Failed rules:")
        for fr in result["failed_rules"]:
            print(f"    [{fr['rule']}] {fr['name']}: {fr['detail']}")
    
    print()

## 3. Constraint Trace Deep Dive

Every configuration carries its full 14-rule trace. Let's look at Scenario 1's recommended config.

In [ ]:
req = scenarios[0][1]
result = solver.solve_with_explanation(req)

if result["status"] == "solved":
    trace = result["recommended"]["constraint_trace"]
    hinge = result["recommended"]["candidates"]["hinge"]
    plate = result["recommended"]["candidates"]["plate"]
    print(f"Configuration: {hinge['sku']} + {plate['sku']}\n")
    print(f"{'Rule':<6} {'Name':<28} {'Result':<6} Detail")
    print("-" * 90)
    for rule in trace:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"{rule['rule']:<6} {rule['name']:<28} {icon:<6} {rule['detail']}")

## 4. All Valid Configurations — No Brand Preference

With brand lock disabled and no preferred brand, the solver evaluates all 2,915 hinge × plate pairs.

In [ ]:
open_req = HingeRequirements(
    cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
    door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
    boring_pattern_mm=45, soft_close=False,
)

start = time.perf_counter()
all_valid = solver.solve(open_req)
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Found {len(all_valid)} valid configurations in {elapsed_ms:.1f}ms\n")

# Show by brand
brand_dist = Counter(c.candidates["hinge"].brand for c in all_valid)
print(f"By brand: {dict(brand_dist)}\n")

print(f"{'#':>3}  {'Hinge SKU':<22} {'Plate SKU':<22} {'Brand':<8} {'Hinges':>6} {'Price/Door':>11}")
print("-" * 75)
for i, config in enumerate(all_valid, 1):
    h = config.candidates["hinge"]
    p = config.candidates["plate"]
    price = f"${config.total_price_usd:.2f}" if config.total_price_usd else "N/A"
    print(f"{i:>3}  {h.sku:<22} {p.sku:<22} {h.brand:<8} {config.derived['hinges_per_door']:>6} {price:>11}")

## 5. Pre-filter Impact

The N-candidate solver applies pre-filters before the Cartesian product. For hinges, this narrows by application, cabinet type, and brand — dramatically reducing the search space.

In [ ]:
# Compare filtered vs unfiltered search space
test_reqs = [
    ("Full overlay, frameless, no brand", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
        door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
        boring_pattern_mm=45, soft_close=False)),
    ("Full overlay, frameless, Blum", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
        door_weight_kg=5.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
        boring_pattern_mm=45, soft_close=False, preferred_brand="Blum")),
    ("Half overlay, frameless, Blum", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=720,
        door_weight_kg=4.0, application=ApplicationType.HALF_OVERLAY, desired_overlay_mm=6,
        boring_pattern_mm=45, soft_close=False, preferred_brand="Blum")),
    ("Corner, frameless, no brand", HingeRequirements(
        cabinet_type=CabinetType.FRAMELESS, door_thickness_mm=19, door_height_mm=800,
        door_weight_kg=4.0, application=ApplicationType.FULL_OVERLAY, desired_overlay_mm=16,
        boring_pattern_mm=45, soft_close=False, cabinet_position=CabinetPosition.CORNER)),
]

full_product = len(hinges) * len(plates)
print(f"Full Cartesian product: {len(hinges)} hinges x {len(plates)} plates = {full_product:,} pairs\n")
print(f"{'Scenario':<40} {'Filtered hinges':>16} {'Pairs evaluated':>16} {'Reduction':>10}")
print("-" * 85)

for label, req in test_reqs:
    filtered = solver._apply_pre_filters(req)
    n_hinges = len(filtered["hinge"])
    n_pairs = n_hinges * len(filtered["plate"])
    reduction = (1 - n_pairs / full_product) * 100
    print(f"{label:<40} {n_hinges:>16} {n_pairs:>16,} {reduction:>9.0f}%")

## 6. Performance

The full 53 × 55 catalog with 14 rules per pair. Even without pre-filtering, this completes in milliseconds.

In [ ]:
bench_scenarios = [
    ("Standard (Blum, full overlay)", scenarios[0][1]),
    ("Corner cabinet (all brands)", scenarios[1][1]),
    ("Tall pantry (Grass)", scenarios[2][1]),
    ("Adjacent doors (Blum, half)", scenarios[3][1]),
    ("No brand preference", open_req),
]

print(f"{'Scenario':<35} {'Time (ms)':>10} {'Valid':>6} {'Pairs evaluated':>16}")
print("-" * 70)
for label, req in bench_scenarios:
    filtered = solver._apply_pre_filters(req)
    n_pairs = len(filtered["hinge"]) * len(filtered["plate"])
    
    start = time.perf_counter()
    results = solver.solve(req)
    elapsed = (time.perf_counter() - start) * 1000
    
    print(f"{label:<35} {elapsed:>9.1f}ms {len(results):>6} {n_pairs:>16,}")

## Summary

| Aspect | Concealed Hinges (N=2) |
|---|---|
| **Roles** | hinge, plate |
| **Cartesian product** | hinges × plates |
| **Rules** | 14 (13 hard, 1 preference) |
| **Catalog** | 53 hinges × 55 plates = 2,915 pairs (real data) |
| **Pre-filtering** | Application, cabinet type, brand (reduces to ~5-15 hinges) |
| **Data source** | `sample-data/hinges.json`, `sample-data/mounting_plates.json` |

The same `NCandidateSolver` also handles:
- **N=1** drawer slides (4 products, no pairing) — see `v2_drawer_slide_demo.ipynb`
- **N=3** LED lighting (5 × 4 × 4 = 80 triples) — see `../v2-n-candidate/v2_n_candidate_demo.ipynb`

**Source code:** `engine_v2/core/solver_n.py` (solver), `engine_v2/families/concealed_hinge/` (models, rules, loader)